### Preparation

In [ ]:
!wget -c https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/weekly/clinvar_20251027.vcf.gz
!wget -c https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.26_GRCh38/GCF_000001405.26_GRCh38_genomic.fna.gz

In [1]:
import random
import gzip
import numpy as np
import pickle
from tqdm import tqdm
from pyfastx import Fasta

In [ ]:
# Load genome
genome = Fasta("GCF_000001405.26_GRCh38_genomic.fna.gz")
seq_map = {}
for seq in genome:
    description = seq.description
    if description.endswith("Primary Assembly"):
        chrom = description.split(",")[0].split(" ")[-1]
        seq_map[chrom] = {"chrom": seq.name, "seq": seq.seq.upper()}

In [ ]:
# Get categories
categories = ["Benign", "Likely_benign", "Pathogenic", "Likely_pathogenic", "Uncertain_significance"]
variations = {"SNVs": {}, "Indels": {}}
with gzip.open("clinvar_20251027.vcf.gz", "rt") as invcf:
    for line in invcf:
        if line.startswith("#"):
            continue
        info = line.strip().split("\t")
        chrom = info[0]
        pos = int(info[1])
        ref_id = info[2]
        ref = info[3]
        alt = info[4]
        reflen = len(ref)
        altlen = len(alt)
        if reflen == altlen and reflen == 1:
            muttype = "SNVs"
        else:
            muttype = "Indels"
        mutinfo = {x.split("=")[0]: x.split("=")[1] for x in info [-1].split(";")}
        allele_id = mutinfo["ALLELEID"] if "ALLELEID" in mutinfo else "NA"
        classification = mutinfo["CLNSIG"] if "CLNSIG" in mutinfo else "Unknown"
        if classification in categories:
            if classification not in variations[muttype]:
                variations[muttype][classification] = []
            variations[muttype][classification].append([ref_id, chrom, pos, ref, alt])

# Random sampling for each type
number = 1000
window_size = 2048
random.seed(42)
with open("clinvar_selected_variations.csv", "w") as outf:
    print("seqid", "chrom", "pos", "type", "ref", "alt",
          "raw_seq", "seq_ref", "seq_mut", "label", sep=",", file=outf)
    for muttype in variations:
        for classification in tqdm(categories, desc=muttype):
            variants = random.sample(variations[muttype][classification], int(number * 1.2))
            count = 0
            for varinfo in variants:
                if count >= number:
                    break
                ref_id = varinfo[0]
                chrom = varinfo[1]
                pos = int(varinfo[2])
                ref = varinfo[3]
                alt = varinfo[4]
                mid = window_size // 2
                start = pos - mid
                if start < 0:
                    continue
                delta = len(alt) - len(ref)
                if delta > 0:
                    ref_len = window_size - delta
                else:
                    ref_len = window_size
                end_ref = start + ref_len
                if chrom not in seq_map:
                    continue
                raw = seq_map[chrom]["seq"][start:start+window_size]
                seq = seq_map[chrom]["seq"][start:end_ref]
                if len(seq) != ref_len:
                    continue
                ref_allele_start_index = mid - 1
                ref_allele_end_index = ref_allele_start_index + len(ref)
                if ref_allele_end_index > len(seq):
                    continue
                prefix = seq[:ref_allele_start_index]
                suffix = seq[ref_allele_end_index:]
                seq_mut = prefix + alt + suffix
                if "." in seq or "." in seq_mut:
                    continue
                if "N" in seq.upper() or "N" in seq_mut.upper():
                    continue
                if len(seq) < 100 or len(seq_mut) < 100:
                    continue
                print(ref_id, chrom, pos, muttype, ref, alt, raw,
                      seq, seq_mut, classification, sep=",", file=outf)
                count += 1

### Inference

In [ ]:
from dnallm import load_config
from dnallm import load_model_and_tokenizer, Mutagenesis

In [ ]:
# Load configurations
configs = load_config("./inference_config.yaml")
configs["task"].task_type = "generation"
configs["inference"].max_length = 2050

In [ ]:
# Load model and tokenizer
# model_name = "lgq12697/JanusDNA-wo-midattn-144dim-mlp"
model_name = "lgq12697/hyenadna-medium-160k-seqlen-hf"
# model_name = "lgq12697/evo2_1b_base"
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs["task"], source="modelscope")
tokenizer.model_max_length = configs["inference"].max_length

In [ ]:
# Get sequences for scoring
def rev_comp(seq):
    basemap = {"A": "T", "T": "A",
               "C": "G", "G": "C",
               "N": "N"}
    new_seq = "".join([basemap[b] for b in seq][::-1])
    return new_seq

raw_seqs1 = []
raw_seqs2 = []
seq_labels = {}
with open("clinvar_selected_variations.csv") as infile:
    for line in tqdm(infile):
        if line.startswith("seqid"):
            continue
        info = line.strip().split(",")
        ref_id = info[0]
        muttype = info[3]
        label = info[-1]
        raw_seq1 = info[6]
        raw_seq2 = rev_comp(raw_seq1)
        raw_seqs1.append(raw_seq1)
        raw_seqs2.append(raw_seq2)
        seq_labels[ref_id] = [muttype, label]

In [ ]:
# Initialize mutagenesis instance
mutagenesis = Mutagenesis(
    model=model, tokenizer=tokenizer, config=configs
)

In [ ]:
window_size = 2048
mid = window_size // 2
all_scores = {}
for i, ref_id in enumerate(seq_labels):
    muttype, label = seq_labels[ref_id]
    mutagenesis.sequences = {"sequence": [raw_seqs1[i], raw_seqs2[i]]}
    scores = mutagenesis.clm_evaluate(return_sum=False)
    if label not in all_scores:
        all_scores[label] = []
    # average scores from fwd and rev strand
    all_scores[label].append(np.array(scores[0]) + np.array(scores[1]) / 2)

In [ ]:
# calculate average score
average_score = {}
for key in all_scores:
    average_score[key] = -np.mean(all_scores[key], axis=0)[mid-256:mid+256]

# save average scores for convenience
# with open("average_scores.pkl", "wb") as f:
#     pickle.dump(average_score, f)

In [ ]:
from dnallm.inference.plot import plot_line

# Load average scores if needed
# with open("average_scores.pkl", "rb") as f:
#     average_score = pickle.load(f)

# Plot line chart
chart = plot_line(average_score, width=400, height=400)
chart

alt.Chart(...)